In [ ]:
def gen_barcode_mrid(patternLengths, c2c_dists, barcode_length=4500, barcode_start=1000):
    
    barcode = np.ones((1000,barcode_length))
    # l=barcode_start
    # ticks=[l]

    a = np.sum(c2c_dists)
    b = patternLengths[-1]/2
    c = patternLengths[0]/2
    l = int(barcode_length - (a+b+c))
    ticks=[l]
    
    for i, bar in enumerate(patternLengths):
        if i == 0:
        # if l==barcode_start:
            barInt=np.round(bar).astype(int)
            barcode[:, l:l+barInt]=0
            ticks.append(l+barInt/2)
            ticks.append(l+barInt)
            l=l+np.round(bar/2).astype(int)+np.round(c2c_dists[i]).astype(int)
        else:
            barInt=np.round(bar/2).astype(int)
            barcode[:, l-barInt:l+barInt]=0
            ticks.append(int(l-barInt))
            ticks.append(l)
            ticks.append(l+barInt)
            if i<len(c2c_dists):
                l=l+np.round(c2c_dists[i]).astype(int)
                
    ticks=np.array(ticks)
    tickLabels=(ticks-int(barcode_length - (a+b+c))).astype(int)
    
    return barcode, ticks, tickLabels

## LEGACY CODE
def gen_barcode_wafer(patternPoints, barcode_start=1000, barcode_length=4500):
    num_patterns=int(len(patternPoints)/3)
    barcode = np.ones((1000,barcode_length))
    l=barcode_start

    for pattern in range(num_patterns):
        p1,p2,p3=patternPoints[3*pattern:3*(pattern+1)]
        barcode[:, int(l+p1):int(l+p3+1)]=0

    return barcode


def equalize_barcodes(barcode1, barcode2):
    if np.shape(barcode1)[1]>np.shape(barcode2)[1]:
        diff=np.shape(barcode1)[1]-np.shape(barcode2)[1]
        padd=np.ones((np.shape(barcode2)[0], diff))
        barcode2=np.concatenate((padd, barcode2), axis=1)
    elif np.shape(barcode1)[1]<np.shape(barcode2)[1]:
        diff=np.shape(barcode2)[1]-np.shape(barcode1)[1]
        padd=np.ones((np.shape(barcode1)[0], diff))
        barcode1=np.concatenate((padd, barcode1), axis=1)
    
    return barcode1, barcode2
def barcode_cc(barcode_fix, barcode_mvng):
#     barcode_fix, barcode_mvng = equalize_barcodes(barcode_fix, barcode_mvng)
    stride=1
    cc=[]
    barcode_fix=1-barcode_fix
    barcode_fix=barcode_fix[0,:]
#     print("zero padding: "+str(np.shape(barcode_mvng)[-1]))
    barcode_fix=np.append(np.zeros((np.shape(barcode_mvng)[-1])), barcode_fix)
    barcode_fix=np.append(barcode_fix, np.zeros((np.shape(barcode_mvng)[-1])))
    
    barcode_mvng=1-barcode_mvng
    barcode_mvng=barcode_mvng[0,:]
    for i in range(len(barcode_fix)-len(barcode_mvng)):
        corr=np.multiply(barcode_fix[stride*i:stride*i+len(barcode_mvng)], barcode_mvng)
        cc.append(np.sum(corr))    
    
    return cc
## LEGACY CODE

In [ ]:
def barcode_probability(ref_barcodes, test_array, mrid_dict, sigma=None):
    """
    Compute similarity probabilities between reconstructed MRI-barcode and reference MRI-barcodes from designs
    using Manhattan distance based similarities and Gaussian weighting.
    If sigma is not provided, it is estimated from pairwise distances among reference arrays.

    Parameters
    ----------
    reference_arrays : list or array-like
        List of 1D numpy arrays (all same length).
    test_array : np.ndarray
        1D numpy array of same length as reference arrays.
    sigma : float, optional
        Spread parameter controlling how sharply probabilities decay with distance.
        If None, sigma is estimated from the spread of reference arrays.

    Returns
    -------
    probs : np.ndarray
        Probabilities indicating similarity of test_array to each reference array.
    similarities : np.ndarray
        Raw normalized similarities test_array and each reference array.
    sigma : float
        The sigma value used (estimated or provided).
    """
    reference_arrays = []
    for mrid in ref_barcodes:
        c2c = get_centomass(mrid_dict[mrid]["dimensions"], mrid_dict[mrid]["intersegment_distances"])[0]
        barcode, _, _ = gen_barcode_mrid(mrid_dict[mrid]["dimensions"][:,-1], c2c)
        reference_arrays.append(barcode[0,:])
        
    reference_arrays = np.array(reference_arrays)
    n_refs = len(reference_arrays)

    # --- Estimate sigma from reference arrays if not provided ---
    if sigma is None:
        # Compute all pairwise normalized Manhattan distances between reference arrays
        dists = []
        for i in range(n_refs):
            for j in range(i + 1, n_refs):
                # d = np.linalg.norm(reference_arrays[i] - reference_arrays[j])
                d = np.sum(reference_arrays[i] == reference_arrays[j])/len(reference_arrays[i])
                dists.append(d)
        dists = np.array(dists)

        sigma = np.std(dists)
        
    # --- Compute squared distances between test and reference arrays ---
    # squared_distances = np.array([np.sum((ref - test_array) ** 2) for ref in reference_arrays])
    similarities = np.array([np.sum(ref == test_array)/len(test_array) for ref in reference_arrays])

    # --- Gaussian weighting ---
    weights = np.exp(similarities / (2 * sigma**2))
    probs = weights / np.sum(weights)

    return probs, similarities, sigma
